In [1]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [2]:
djia = pd.read_csv("djia_news copy.csv")
nasdaq = pd.read_csv("nasdaq.csv")

In [3]:
df = pd.concat([djia, nasdaq], axis=0).reset_index(drop=True)


In [4]:
print(df.head())


   Label Ticker                                           Headline
0      0    MMM  Employer who stole nearly $3M in wages from 15...
1      1    MMM  Huge new Facebook data leak exposed intimate d...
2      0    MMM  A campaign has accelerated to turn a disused r...
3      1    MMM  Google launches global human trafficking helpl...
4      1    MMM  Over 3m Saudi Women Don’t Have ID Cards; Saudi...


In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15562 entries, 0 to 15561
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Label     15562 non-null  int64 
 1   Ticker    15562 non-null  object
 2   Headline  15562 non-null  object
dtypes: int64(1), object(2)
memory usage: 364.9+ KB
None


In [6]:
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

In [7]:
import nltk
nltk.download('stopwords')
nltk.download('vader_lexicon')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gadhv\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\gadhv\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [8]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|[^a-z\s]", "", text)
    text = " ".join([w for w in text.split() if w not in stop_words])
    return text

df["clean_headline"] = df["Headline"].apply(clean_text)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gadhv\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
df["Label"] = df["Label"].astype(int)


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_text = df["clean_headline"]
y = df["Label"]

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words="english"
)

X = vectorizer.fit_transform(X_text)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
print("Logistic Regression Results")
print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))


Logistic Regression Results
              precision    recall  f1-score   support

           0       0.66      0.85      0.74      1630
           1       0.42      0.22      0.29       867
           2       0.00      0.00      0.00        55

    accuracy                           0.62      2552
   macro avg       0.36      0.36      0.34      2552
weighted avg       0.56      0.62      0.57      2552

[[1379  251    0]
 [ 673  193    1]
 [  42   13    0]]


In [13]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)
print("Naive Bayes Results")
print(classification_report(y_test, y_pred_nb))


Naive Bayes Results
              precision    recall  f1-score   support

           0       0.65      0.92      0.76      1630
           1       0.45      0.13      0.21       867
           2       0.00      0.00      0.00        55

    accuracy                           0.63      2552
   macro avg       0.37      0.35      0.32      2552
weighted avg       0.57      0.63      0.56      2552



c:\Users\gadhv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\gadhv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\gadhv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest Results")
print(classification_report(y_test, y_pred_rf))


Random Forest Results
              precision    recall  f1-score   support

           0       0.67      0.85      0.75      1630
           1       0.47      0.25      0.32       867
           2       0.20      0.05      0.09        55

    accuracy                           0.63      2552
   macro avg       0.45      0.38      0.39      2552
weighted avg       0.59      0.63      0.59      2552



In [15]:
import joblib

joblib.dump(lr, "news_sentiment_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


['tfidf_vectorizer.pkl']

In [16]:
def predict_sentiment(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])
    pred = lr.predict(vec)[0]
    return "Positive News" if pred == 1 else "Negative News"

#print(predict_sentiment("Apple shares surge after strong quarterly earnings"))
print(predict_sentiment("Reliance Industries stock guaranteed to double in one day, insiders claim"))



Negative News


In [17]:
# Install yfinance first
# pip install yfinance

import yfinance as yf

ticker = "AAPL"  # choose the stock you want
start_date = "2015-01-01"
end_date = "2025-01-01"

df = yf.download(ticker, start=start_date, end=end_date, interval="1d")

df.to_csv("AAPL_historical_data.csv")
print("Saved AAPL_historical_data.csv")


[*********************100%***********************]  1 of 1 completed

Saved AAPL_historical_data.csv


In [20]:
print(prices.columns)
print(prices.head())

Index(['date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')
         date      Close       High        Low       Open     Volume
0  2015-01-02  24.237549  24.705318  23.798599  24.694233  212818400
1  2015-01-05  23.554737  24.086797  23.368517  24.006988  257142000
2  2015-01-06  23.556961  23.816340  23.195602  23.619034  263188400
3  2015-01-07  23.887283  23.987044  23.654506  23.765352  160423600
4  2015-01-08  24.805088  24.862728  24.097891  24.215389  237458000


In [22]:
import pandas as pd

prices = pd.read_csv("AAPL_historical_data.csv", skiprows=[1,2])
prices.rename(columns={"Price": "date"}, inplace=True)

prices["date"] = pd.to_datetime(prices["date"])  # let pandas infer format
prices = prices.sort_values("date")

print(prices.head())
print(prices.columns)


        date      Close       High        Low       Open     Volume
0 2015-01-02  24.237549  24.705318  23.798599  24.694233  212818400
1 2015-01-05  23.554737  24.086797  23.368517  24.006988  257142000
2 2015-01-06  23.556961  23.816340  23.195602  23.619034  263188400
3 2015-01-07  23.887283  23.987044  23.654506  23.765352  160423600
4 2015-01-08  24.805088  24.862728  24.097891  24.215389  237458000
Index(['date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')


In [32]:
news_djia = pd.read_csv("djia_news copy.csv")
news_nasdaq = pd.read_csv("nasdaq.csv")

news = pd.concat([news_djia, news_nasdaq], axis=0)
news.dropna(inplace=True)

news["clean_headline"] = news["Headline"].apply(clean_text)
news["sentiment"] = news["clean_headline"].apply(
    lambda x: lr.predict(vectorizer.transform([x]))[0]
)

print(news.head())


   Label Ticker                                           Headline  \
0      0    MMM  Employer who stole nearly $3M in wages from 15...   
1      1    MMM  Huge new Facebook data leak exposed intimate d...   
2      0    MMM  A campaign has accelerated to turn a disused r...   
3      1    MMM  Google launches global human trafficking helpl...   
4      1    MMM  Over 3m Saudi Women Don’t Have ID Cards; Saudi...   

                                      clean_headline  sentiment  
0          employer stole nearly wages workers fined          0  
1  huge new facebook data leak exposed intimate d...          1  
2  campaign accelerated turn disused railway line...          0  
3  google launches global human trafficking helpl...          1  
4  saudi women dont id cards saudi grand mufti sa...          0  


In [33]:
apple_news = news[news["Ticker"] == "AAPL"]
print("Apple news samples:")
print(apple_news.head())


Apple news samples:
     Label Ticker                                           Headline  \
192      1   AAPL  Apple the world's most profitable firm has a s...   
193      0   AAPL  Apple bows to China by censoring Taiwan flag e...   
194      0   AAPL  France says 'crazy' that Apple and others get ...   
195      0   AAPL  Apple cofounder Steve Wozniak says most people...   
196      0   AAPL  Shouting ‘pay your taxes’ activists occupy App...   

                                        clean_headline  sentiment  
192  apple worlds profitable firm secretive new str...          0  
193       apple bows china censoring taiwan flag emoji          0  
194  france says crazy apple others get permanent t...          1  
195  apple cofounder steve wozniak says people figu...          0  
196  shouting pay taxes activists occupy apple reta...          0  


In [34]:
sentiment_score = apple_news["sentiment"].mean()
print("Overall Apple sentiment score:", sentiment_score)


Overall Apple sentiment score: 0.25862068965517243


In [36]:
prices = pd.read_csv("AAPL_historical_data.csv", header=None)

# Fix the weird header structure
prices.columns = prices.iloc[0]
prices = prices.drop(index=[0, 1, 2]).reset_index(drop=True)

# Rename first column to Date
prices.rename(columns={prices.columns[0]: "Date"}, inplace=True)

# Convert datatypes
prices["Date"] = pd.to_datetime(prices["Date"])
prices[["Close", "High", "Low"]] = prices[["Close", "High", "Low"]].astype(float)

prices = prices.sort_values("Date")

print(prices.head())
print(prices.columns)


0       Date      Close       High        Low                Open     Volume
0 2015-01-02  24.237549  24.705318  23.798599  24.694233092060063  212818400
1 2015-01-05  23.554737  24.086797  23.368517  24.006988246177652  257142000
2 2015-01-06  23.556961  23.816340  23.195602  23.619034466383983  263188400
3 2015-01-07  23.887283  23.987044  23.654506   23.76535220011818  160423600
4 2015-01-08  24.805088  24.862728  24.097891  24.215388908830796  237458000
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name=0)


In [39]:
prices = pd.read_csv("AAPL_historical_data.csv", header=None)

# Step 1: Use first row as column names
prices.columns = prices.iloc[0]

# Step 2: Drop non-data rows
prices = prices.drop(index=[0, 1, 2]).reset_index(drop=True)

# Step 3: Rename first column to Date
prices.rename(columns={prices.columns[0]: "Date"}, inplace=True)

# Step 4: Convert datatypes safely
prices["Date"] = pd.to_datetime(prices["Date"], errors="coerce")
for col in ["Close", "High", "Low"]:
    prices[col] = pd.to_numeric(prices[col], errors="coerce")

prices.dropna(inplace=True)
prices = prices.sort_values("Date")

print(prices.dtypes)
print(prices.head())


0
Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open              object
Volume            object
dtype: object
0       Date      Close       High        Low                Open     Volume
0 2015-01-02  24.237549  24.705318  23.798599  24.694233092060063  212818400
1 2015-01-05  23.554737  24.086797  23.368517  24.006988246177652  257142000
2 2015-01-06  23.556961  23.816340  23.195602  23.619034466383983  263188400
3 2015-01-07  23.887283  23.987044  23.654506   23.76535220011818  160423600
4 2015-01-08  24.805088  24.862728  24.097891  24.215388908830796  237458000


In [40]:
prices["return"] = prices["Close"].pct_change()
prices["target"] = (prices["return"].shift(-1) > 0).astype(int)
prices.dropna(inplace=True)

print(prices[["Date", "Close", "return", "target"]].head())


0       Date      Close    return  target
1 2015-01-05  23.554737 -0.028172       1
2 2015-01-06  23.556961  0.000094       1
3 2015-01-07  23.887283  0.014022       1
4 2015-01-08  24.805088  0.038422       1
5 2015-01-09  24.831678  0.001072       0


In [41]:
prices["sentiment_score"] = sentiment_score


In [42]:
prices["ma_5"] = prices["Close"].rolling(5).mean()
prices["ma_10"] = prices["Close"].rolling(10).mean()
prices["volatility"] = prices["return"].rolling(5).std()

prices.dropna(inplace=True)


In [43]:
features = ["sentiment_score", "return", "ma_5", "ma_10", "volatility"]
X = prices[features]
y = prices["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

from sklearn.metrics import classification_report
print(classification_report(y_test, model.predict(X_test)))


              precision    recall  f1-score   support

           0       0.45      0.86      0.59       220
           1       0.63      0.19      0.29       282

    accuracy                           0.48       502
   macro avg       0.54      0.52      0.44       502
weighted avg       0.55      0.48      0.42       502



In [44]:
prices["next_close"] = prices["Close"].shift(-1)
prices.dropna(inplace=True)


In [45]:
features = ["sentiment_score", "return", "ma_5", "ma_10", "volatility"]
X = prices[features]
y = prices["next_close"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)


In [46]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

reg_model = RandomForestRegressor(n_estimators=300, random_state=42)
reg_model.fit(X_train, y_train)

y_pred = reg_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))


MAE: 19.890295590561227
R² Score: -0.13452652036821577


In [49]:
#Predict Tomorrow’s Stock Price Using News

def predict_next_price(news_text):
    clean = clean_text(news_text)
    sentiment = lr.predict(vectorizer.transform([clean]))[0]

    latest = prices.iloc[-1][features].copy()
    latest["sentiment_score"] = sentiment

    latest_df = pd.DataFrame([latest], columns=features)
    pred_price = reg_model.predict(latest_df)[0]

    return pred_price


print("Predicted next close price:",
      predict_next_price("Apple announces record-breaking iPhone sales and AI expansion"))


Predicted next close price: 170.75555826822918
